In [2]:
# Experiment 4: Tokenizer lab - BPE vs WordPiece vs SentencePiece on English, Telugu and Hindi
from transformers import AutoTokenizer
from google.colab import userdata
from google import genai
import pandas as pd

texts = {
    "English": "I am going to school.",
    "Telugu":  "నేను బడికి వెళ్తున్నాను.",
    "Hindi":   "मैं स्कूल जा रहा हूँ ।",
}
tokenizers = {
    "BPE (gpt2)": "gpt2",
    "WordPiece (bert-base-multilingual-cased)": "bert-base-multilingual-cased",
    "SentencePiece (xlm-roberta-base)": "xlm-roberta-base",
}

rows = []
for name, model_id in tokenizers.items():
    tok = AutoTokenizer.from_pretrained(model_id)
    for lang, s in texts.items():
        ids = tok(s, add_special_tokens=False).input_ids
        rows.append({"tokenizer": name, "language": lang, "words": len(s.split()),
                     "tokens": len(ids), "tokens_per_word": round(len(ids) / len(s.split()), 2)})
    print(name, "| vocab size:", tok.vocab_size)
    print("   English pieces:", tok.tokenize(texts["English"])[:10])
    print("   Telugu pieces :", tok.tokenize(texts["Telugu"])[:10])
    print("   Token IDs (Hindi):", tok(texts["Hindi"], add_special_tokens=False).input_ids[:10])

# Hosted model's own tokenizer (Gemini) for comparison
client = genai.Client(api_key=userdata.get("gemini_api_key"))
for lang, s in texts.items():
    n = client.models.count_tokens(model="gemini-3.1-flash-lite", contents=s).total_tokens
    rows.append({"tokenizer": "Gemini (gemini-3.1-flash-lite)", "language": lang, "words": len(s.split()),
                 "tokens": n, "tokens_per_word": round(n / len(s.split()), 2)})

df = pd.DataFrame(rows)
print(df.to_string(index=False))

# Token-ratio table (documented result): how many times more tokens than English
ratio = df.pivot(index="tokenizer", columns="language", values="tokens")
ratio["Telugu / English"] = (ratio["Telugu"] / ratio["English"]).round(2)
ratio["Hindi / English"] = (ratio["Hindi"] / ratio["English"]).round(2)
print("\nToken-ratio table:")
print(ratio.to_string())

BPE (gpt2) | vocab size: 50257
   English pieces: ['I', 'Ġam', 'Ġgoing', 'Ġto', 'Ġschool', '.']
   Telugu pieces : ['à', '°', '¨', 'à', '±', 'ĩ', 'à', '°', '¨', 'à']
   Token IDs (Hindi): [11976, 106, 24231, 230, 11976, 224, 28225, 116, 24231, 235]
WordPiece (bert-base-multilingual-cased) | vocab size: 119547
   English pieces: ['I', 'am', 'going', 'to', 'school', '.']
   Telugu pieces : ['న', '##ే', '##ను', 'బ', '##డి', '##కి', 'వ', '##ె', '##ళ', '##్']
   Token IDs (Hindi): [889, 99007, 86709, 24562, 36475, 899, 15778, 28462, 920]
SentencePiece (xlm-roberta-base) | vocab size: 250002
   English pieces: ['▁I', '▁am', '▁going', '▁to', '▁school', '.']
   Telugu pieces : ['▁నేను', '▁', 'బడి', 'కి', '▁వెళ్త', 'ున్న', 'ాను', '.']
   Token IDs (Hindi): [10399, 54585, 5006, 7172, 28035, 207]
                               tokenizer language  words  tokens  tokens_per_word
                              BPE (gpt2)  English      5       6             1.20
                              BPE (gpt2